UNet - perception in v2 - SR

In [ ]:
from fastai.vision.all import *
import cv2
import numpy
import os
import pathlib
import torch
import fastai; fastai.__version__

In [ ]:
torch.cuda.set_device(0)
torch.cuda.current_device(), torch.cuda.get_device_name(0)

In [ ]:
path = Path.cwd()/'data'

In [ ]:
path_LR = path
subfolders = ['x2']

In [ ]:
# Set seeds for reproducibility
%run util/set_seeds.ipynb

# run Dataloader
%run util/Diffusion_Dataloader_train.ipynb

# run notebook to set up feature loss 
%run util/Diffusion_Perception_loss.ipynb

In [ ]:
dls_den, dblock = get_dls(bs=32) # test with smaller batch size for now

In [ ]:
dblock.summary(path_LR) # check pipelines are working as expected 

In [ ]:
loss_func = FeatureLoss(vgg_m, blocks,[125], [1,1,1,1,1], [1,1,1,1,1])

In [ ]:
def create_gen_learner_others():
    return unet_learner(dls_den, bbone, loss_func=loss_func, blur=True, norm_type=NormType.Weight, 
                        self_attention=True, y_range=y_range, pretrained=True, 
                        metrics=metrics, cbs=cbs
                        )

In [ ]:
inputs = ['x2']
losses = ['Feat_0','Feat_1','Feat_2','Feat_3','Feat_4']
for j in inputs:
    path_LR = path/j        
    dls_den, dblock = get_dls(bs=120)    # check if your GPU can handle this batch size, otherwise reduce it
    for i in losses:
        if '0' in i:
            fname='SR_R18_Feat0'
            loss_func = FeatureLoss(vgg_m, blocks,[125], [92,0,0,0,0], [0,0,0,0,0])
        if '1' in i:
            fname='SR_R18_Feat1'
            loss_func = FeatureLoss(vgg_m, blocks,[125], [0,52,0,0,0], [0,0,0,0,0])
        if '2' in i:
            fname='SR_R18_Feat2'
            loss_func = FeatureLoss(vgg_m, blocks,[125], [0,0,46,0,0], [0,0,0,0,0])
        if '3' in i:
            fname='SR_R18_Feat3'
            loss_func = FeatureLoss(vgg_m, blocks,[125], [0,0,0,94,0], [0,0,0,0,0])
        if '4' in i:
            fname='SR_R18_Feat4'
            loss_func = FeatureLoss(vgg_m, blocks,[125], [0,0,0,0,108], [0,0,0,0,0])
        metrics = LossMetrics(loss_func.metric_names)
        cbs=CSVLogger(fname=fname+'.csv')
        learn_den = create_gen_learner_others()  
        #learn_den.lr_find()
        learn_den.fine_tune(100,1e-4,wd=wd,freeze_epochs=5)
        learn_den.save(fname)